In [ ]:
"""
Script 2: Blind Performance Evaluation and Benchmarking
This standalone script strictly computes the Performance Indicators (RMSE, sMAPE and DA)
using ONLY the blind test set actuals (extracted directly from the raw data)
and the pre-computed model predictions.
It evaluates the 10 models, including FMFRA, across 4 coastal cities.
"""

import pandas as pd
import numpy as np
import os

# ---------------------------------------------------------
# METRIC DEFINITIONS
# ---------------------------------------------------------
def calculate_rmse(actual, predicted):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean((actual - predicted)**2))

def calculate_smape(actual, predicted):
    """Symmetric Mean Absolute Percentage Error (%)"""
    denominator = (np.abs(actual) + np.abs(predicted)) / 2.0
    diff = np.abs(actual - predicted) / denominator
    diff[denominator == 0] = 0.0  # Handle division by zero
    return np.mean(diff) * 100

def calculate_da(actual, predicted):
    """
    Directional Accuracy (DA) (%)
    Measures the model's ability to forecast the correct direction of change.
    """
    a_diff = np.sign(np.diff(actual))
    p_diff = np.sign(np.diff(predicted))
    correct_directions = np.sum(a_diff == p_diff)
    total_transitions = len(a_diff)
    return (correct_directions / total_transitions) * 100

# ---------------------------------------------------------
# DATA LOADING EXTRACTION (PREVENTING DATA LEAKAGE)
# ---------------------------------------------------------
def get_blind_test_actuals(city, data_dir="../data"):
    """
    Loads the original raw dataset and strictly extracts the final 10%
    to serve as the pristine ground truth for blind evaluation.
    """
    filepath = f"{data_dir}/{city}_updated_2025.csv"
    df = pd.read_csv(filepath)

    # Strict chronological split: 70% Train, 20% Validation, 10% Test
    n = len(df)
    val_end = int(n * 0.90)

    # Extract only the unseen test portion (last 10%)
    test_df = df.iloc[val_end:].copy()

    return test_df['temperature'].values, test_df['date'].values

# ---------------------------------------------------------
# MAIN EVALUATION PIPELINE
# ---------------------------------------------------------
def main():
    cities = ['TAMPICO', 'MATAMOROS', 'HOUSTON', 'MIAMI']

    # The 10 models benchmarked in the study
    models = ['FMFRA', 'HW', 'SARIMA', 'SSA', 'XGB', 'LGBM', 'LSTM', 'RFR', 'LR', 'CNN']

    # List to store all calculated metrics to build the final benchmarking table
    all_results = []

    print("=========================================================")
    print(" STARTING BLIND PERFORMANCE EVALUATION (TEST SET ONLY) ")
    print("=========================================================\n")

    # Directories
    data_dir = "../data"
    output_dir = "../results"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for city in cities:
        print(f"\n--- Evaluating {city} ---")

        # 1. Load the pristine ground truth directly from raw data
        try:
            actuals, test_dates = get_blind_test_actuals(city, data_dir)
            print(f"Raw Test Set Loaded: {len(actuals)} months extracted for evaluation.")
        except FileNotFoundError:
            print(f"Warning: Raw data file for {city} not found in {data_dir}. Skipping...")
            continue

        # 2. Load the model predictions
        filepath = f"{output_dir}/{city}_test_predictions.csv"
        try:
            pred_df = pd.read_csv(filepath)
            print(f"Predictions Loaded successfully.")
        except FileNotFoundError:
            print(f"Warning: Prediction file {filepath} not found. Skipping...")
            continue

        # 3. Compute metrics for each available model
        for model in models:
            col_name = f"{model}_Prediction"
            if col_name in pred_df.columns:
                predictions = pred_df[col_name].values

                # Ensure the lengths match before calculating
                if len(actuals) != len(predictions):
                    print(f"  -> Error: Length mismatch for {model} in {city}. Actuals: {len(actuals)}, Preds: {len(predictions)}")
                    continue

                # Calculate the 3 standard metrics
                rmse = calculate_rmse(actuals, predictions)
                smape = calculate_smape(actuals, predictions)
                da = calculate_da(actuals, predictions)

                # Append to our master results list
                all_results.append({
                    'City': city,
                    'Model': model,
                    'RMSE': round(rmse, 4),
                    'sMAPE': round(smape, 4),
                    'DA': round(da, 4)
                })
            else:
                print(f"  -> Missing predictions for {model} in {city}")

    # Compile the final Benchmarking Table
    if all_results:
        results_df = pd.DataFrame(all_results)

        print("\n=========================================================")
        print(" OVERALL BENCHMARKING RESULTS (READY FOR STATISTICAL TESTS)")
        print("=========================================================")
        print(results_df.head(15)) # Print a preview

        # Save the comprehensive metrics table for Wilcoxon / Friedman tests
        final_csv_path = f"{output_dir}/Final_Benchmarking_Metrics.csv"
        results_df.to_csv(final_csv_path, index=False)
        print(f"\n✅ All metrics successfully evaluated and saved to: {final_csv_path}")
        print("These results correspond strictly to the unseen 10% test split.")

if __name__ == "__main__":
    main()